# Silver Layer — Tratamento, Deduplicação e Integridade
**Autor**: Laura Virginia Ferreira Soares

**Pipeline:** RPE Sales Data | **Camada:** Silver (Curated)

In [0]:
# etapa 1 — Imports e Configurações
import os
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, DateType, TimestampType, StringType
from pyspark.sql.window import Window

SOURCE_PATH = "/Volumes/workspace/default/rpe_landing/"
CATALOG     = "workspace"

print("Configuracoes OK")

Configuracoes OK


In [0]:
# etapa 2 — Leitura da camada Bronze
bronze_df = spark.table(f"{CATALOG}.default.bronze_sales_raw")
print(f" Registros Bronze (com duplicatas): {bronze_df.count()}")

 Registros Bronze (com duplicatas): 3046


In [0]:
# etapa 3 — Tipagem e padronização
typed_df = (
    bronze_df
    .withColumn("order_id",   F.col("order_id").cast(StringType()))
    .withColumn("product_id", F.col("product_id").cast(IntegerType()))
    .withColumn("quantity",   F.col("quantity").cast(IntegerType()))
    .withColumn("amount",     F.col("amount").cast(DoubleType()))
    .withColumn("status",     F.lower(F.trim(F.col("status"))))
    .withColumn("sale_date",  F.to_date(F.col("sale_date"), "yyyy-MM-dd"))
    .withColumn("discount",   F.col("discount").cast(DoubleType()))
    .withColumn("seller_id",  F.col("_seller_id"))
    .withColumn("ref_year",   F.col("_ref_year"))
    .withColumn("ref_month",  F.col("_ref_month"))
    .withColumn("source_file",F.col("_source_file"))
)

print("Tipagem OK")

Tipagem OK


In [0]:
# etapa 4 — Filtros de qualidade
VALID_STATUSES = ["completed", "cancelled", "pending", "refunded"]

clean_df = (
    typed_df
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("product_id").isNotNull())
    .filter(F.col("seller_id").isNotNull())
    .filter(F.col("sale_date").isNotNull())
    .filter(F.col("amount") > 0)
    .filter(F.col("quantity") > 0)
    .filter(F.col("status").isin(VALID_STATUSES))
)

rejected_count = typed_df.count() - clean_df.count()
print(f" Qualidade OK")
print(f"Registros rejeitados por qualidade: {rejected_count}")

 Qualidade OK
Registros rejeitados por qualidade: 0


In [0]:
# etapa 5 — Deduplicação
dedup_window = Window.partitionBy(
    "order_id", "product_id", "quantity", "amount", "status", "sale_date", "seller_id"
).orderBy(F.col("source_file").asc())

deduped_df = (
    clean_df
    .withColumn("_row_num", F.row_number().over(dedup_window))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)

removed_dups = clean_df.count() - deduped_df.count()
print(f" Deduplicação OK")
print(f" Duplicatas removidas: {removed_dups}")
print(f" Registros após dedup: {deduped_df.count()}")

 Deduplicação OK
 Duplicatas removidas: 144
 Registros após dedup: 2902


In [0]:
# etapa 6 — Leitura das dimensões e integridade referencial
dim_seller = (
    spark.read
    .option("header", "true")
    .csv(f"{SOURCE_PATH}dim_seller.csv")
    .withColumn("seller_id", F.col("seller_id").cast(IntegerType()))
    .withColumn("seller_name", F.initcap(F.trim(F.col("seller_name"))))
    .withColumn("state", F.upper(F.trim(F.col("state"))))
)

dim_product = (
    spark.read
    .option("header", "true")
    .csv(f"{SOURCE_PATH}dim_product.csv")
    .withColumn("product_id", F.col("product_id").cast(IntegerType()))
    .withColumn("product_name", F.initcap(F.trim(F.col("product_name"))))
    .withColumn("category", F.initcap(F.trim(F.col("category"))))
)

valid_seller_ids  = [r.seller_id  for r in dim_seller.collect()]
valid_product_ids = [r.product_id for r in dim_product.collect()]

enriched_df = (
    deduped_df
    .withColumn("seller_registered",  F.col("seller_id").isin(valid_seller_ids))
    .withColumn("product_registered", F.col("product_id").isin(valid_product_ids))
)

print(" Integridade referencial OK")
print("Sellers sem cadastro:")
enriched_df.filter(~F.col("seller_registered")).select("seller_id").distinct().show()
print("Produtos sem cadastro:")
enriched_df.filter(~F.col("product_registered")).select("product_id").distinct().show()

 Integridade referencial OK
Sellers sem cadastro:
+---------+
|seller_id|
+---------+
|        3|
+---------+

Produtos sem cadastro:
+----------+
|product_id|
+----------+
|        10|
|        25|
+----------+



In [0]:
# etapa 7 — verificando se tem venda fora do mes de referencia do arquivo
# ex: arquivo de janeiro com venda registrada em fevereiro
enriched_df = enriched_df.withColumn(
    "late_arriving",
    (F.year(F.col("sale_date"))  != F.col("ref_year")) |
    (F.month(F.col("sale_date")) != F.col("ref_month"))
)

late_count = enriched_df.filter(F.col("late_arriving")).count()

if late_count > 0:
    print(f"atencao: encontrei {late_count} vendas fora do periodo do arquivo")
else:
    print("tudo certo, nenhuma venda fora do periodo esperado")

 Late arriving data: 0 registros identificados


In [0]:
# etapa 8 — Seleção das colunas finais
silver_sales_df = enriched_df.select(
    "order_id",
    "seller_id",
    "product_id",
    "sale_date",
    "quantity",
    "amount",
    "discount",
    "status",
    "ref_year",
    "ref_month",
    "seller_registered",
    "product_registered",
    "late_arriving",
    "source_file",
    F.lit(datetime.utcnow().isoformat()).cast(TimestampType()).alias("silver_ts"),
)

print(f" Registros Silver prontos: {silver_sales_df.count()}")

 Registros Silver prontos: 2902


In [0]:
# etapa 9 — Persistência Silver (saveAsTable)
silver_sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{CATALOG}.default.silver_fact_sales")

print(" silver_fact_sales salva!")

 silver_fact_sales salva!


In [0]:
# etapa 10 — Persistência dimensões Silver
dim_seller.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.default.silver_dim_sellers")

dim_product.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.default.silver_dim_products")

print("silver_dim_sellers salva!")
print("silver_dim_products salva!")

silver_dim_sellers salva!
silver_dim_products salva!


In [0]:
# etapa 11 — final
print("Preview Silver — Fato Vendas:")
spark.sql(f"""
    SELECT
        seller_id,
        ref_year,
        ref_month,
        COUNT(*) AS total_rows,
        ROUND(SUM(amount), 2) AS total_revenue,
        SUM(CASE WHEN status='cancelled' THEN 1 ELSE 0 END) AS cancelled
    FROM {CATALOG}.default.silver_fact_sales
    GROUP BY 1,2,3
    ORDER BY 1,2,3
""").show(50, truncate=False)

print("camada silver concluída com sucesso")

Preview Silver — Fato Vendas:
+---------+--------+---------+----------+-------------+---------+
|seller_id|ref_year|ref_month|total_rows|total_revenue|cancelled|
+---------+--------+---------+----------+-------------+---------+
|1        |2025    |1        |81        |22401.5      |43       |
|1        |2025    |3        |65        |16732.84     |30       |
|1        |2025    |4        |52        |15028.78     |30       |
|1        |2025    |5        |97        |21025.48     |50       |
|1        |2025    |6        |96        |23479.99     |46       |
|1        |2025    |7        |85        |21551.78     |35       |
|1        |2025    |8        |50        |14631.23     |25       |
|1        |2025    |10       |137       |31142.66     |72       |
|1        |2025    |11       |150       |35873.69     |75       |
|3        |2025    |2        |107       |27357.4      |49       |
|3        |2025    |3        |117       |27979.02     |65       |
|3        |2025    |4        |100       |24061